In [ ]:
import os

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os
import scanpy as sc
import plotnine as gg

import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib.colors as mcolors
from tqdm import tqdm


tab10_colors = plt.get_cmap("tab10").colors
tab10_hex = [mcolors.to_hex(c) for c in tab10_colors]
plt.rcParams["svg.fonttype"] = "none"

# Load data

## OPS data

In [ ]:
f_vector_cols = [
    # "N Mismatch_y",
    "Delta time (s)",
    "Instantaneous Growth Rate: Volume",
    "Length",
    "Septum Displacement Length Normalized",
    "Width",
    "mCherry mean_intensity",
]


timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
working_dir = "/workspace/data/Eaton_2025/Data/lDE20_Imaging"
df = pd.read_csv(
    os.path.join(working_dir, "2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv")
)

df_ = df.query("Estimator == 'Mean (Robust)'")
ops_summarystats_df = (
    df_.pivot(index=["sgRNA", "Gene", "N Mismatch"], columns="Variable(s)", values="Value")
    .reset_index()
    .set_index(["sgRNA", "Gene"])
)
ops_summarystats_df


def expand_embeddings(df, columns):
    expanded_dfs = []
    for col in columns:
        if col not in df.columns:
            continue
        col_values = np.stack(df[col].values)
        if col_values.ndim > 2:
            col_values = col_values.reshape(len(df), -1)

        expanded = pd.DataFrame(
            col_values, index=df.index, columns=[f"{col}_{i}" for i in range(col_values.shape[1])]
        )
        expanded_dfs.append(expanded)

    metadata_cols = [c for c in df.columns if c not in columns]
    return pd.concat([df[metadata_cols]] + expanded_dfs, axis=1).reset_index()


df_embeddings = expand_embeddings(timeseries_df, ["Feature Vector"])
tsne_rep = TSNE(n_components=2)
embeddings_tsne = tsne_rep.fit_transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["tsne_x"] = embeddings_tsne[:, 0]
df_embeddings["tsne_y"] = embeddings_tsne[:, 1]
df_embeddings.info()

pca_ = PCA(n_components=2)
pca_.fit(df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]])
embeddings_pca = pca_.transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["pca_x"] = embeddings_pca[:, 0]
df_embeddings["pca_y"] = embeddings_pca[:, 1]
# comparison with growth fitness screen

ops_df = df_embeddings.merge(ops_summarystats_df.reset_index(), on="sgRNA", how="left")
# ops_df

In [ ]:
obsm = dict(features=ops_df[[c for c in ops_df.columns if c.startswith("Feature")]].values)
X_placeholder = np.zeros((len(ops_df), 10))
ops_data = sc.AnnData(obsm=obsm, X=X_placeholder)
sc.pp.neighbors(ops_data, use_rep="features")
sc.tl.umap(ops_data)
sc.tl.leiden(ops_data, key_added="leiden_0.1", resolution=0.1)
sc.tl.leiden(ops_data, key_added="leiden_0.2", resolution=0.2)
sc.tl.leiden(ops_data, key_added="leiden_0.5", resolution=0.5)
sc.tl.leiden(ops_data, key_added="leiden_1.0", resolution=1)

In [ ]:
sc.pl.umap(ops_data, color=["leiden_0.1", "leiden_0.2", "leiden_0.5", "leiden_1.0"])

In [ ]:
ops_df["leiden_0.1"] = ops_data.obs["leiden_0.1"].values
ops_df["leiden_0.2"] = ops_data.obs["leiden_0.2"].values
ops_df["leiden_0.5"] = ops_data.obs["leiden_0.5"].values
ops_df["leiden_1.0"] = ops_data.obs["leiden_1.0"].values

In [ ]:
(
    gg.ggplot(ops_df, gg.aes("tsne_x", "tsne_y", color="leiden_0.2"))
    + gg.geom_point(size=0.2)
    + gg.theme_minimal()
    + gg.theme(
        legend_position="none",
        figure_size=(4, 3),
        dpi=300,
    )
    + gg.labs(
        x="t-SNE 1",
        y="t-SNE 2",
        title="OPS representation",
    )
    + gg.scale_color_manual(values=tab10_hex)
)

In [ ]:
fig = (
    gg.ggplot(ops_df, gg.aes("tsne_x", "tsne_y", color="Instantaneous Growth Rate: Volume"))
    + gg.geom_point(size=0.2)
    + gg.theme_minimal()
    + gg.theme(
        # legend_position="none",
        figure_size=(6.5, 5),
    )
    + gg.labs(
        x="t-SNE 1",
        y="t-SNE 2",
        title="OPS representation",
        color="volumic growth(OPS)",
    )
    # + gg.scale_color_cmap("bwr", limits=[-3.0, 3.0])
)
fig
# fig + gg.theme(legend_position="none", figure_size=(4, 3), dpi=300)

In [ ]:
opssummary_df = ops_df.set_index("sgRNA")[f_vector_cols].copy().dropna(axis=0)
zvals = (opssummary_df - opssummary_df.mean(axis=0)) / opssummary_df.std(axis=0)
tsne_ = TSNE(n_components=2, perplexity=30)
embs = tsne_.fit_transform(zvals)
opssummary_df["opssummary_TSNE1"] = embs[:, 0]
opssummary_df["opssummary_TSNE2"] = embs[:, 1]

In [ ]:
(
    gg.ggplot(opssummary_df, gg.aes("opssummary_TSNE1", "opssummary_TSNE2"))
    + gg.geom_point(size=0.2)
    + gg.theme_minimal()
    + gg.theme(legend_position="none", figure_size=(4, 3), dpi=300)
)

## Transcriptomic data

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

### joint analysis

In [ ]:
transcript_obs_augm = adata_case.obs.copy()
transcript_obs_augm = transcript_obs_augm.merge(
    ops_df, left_on="spacer", right_on="sgRNA", how="left", suffixes=("_tr", "_ops")
)
transcript_obs_augm["leiden_0.5_ops"] = (
    transcript_obs_augm["leiden_0.5_ops"].astype(float).fillna(-1).astype(str)
)
transcript_obs_augm["leiden_0.2_ops"] = (
    transcript_obs_augm["leiden_0.2"].astype(float).fillna(-1).astype(str)
)
transcript_obs_augm["leiden_1.0_ops"] = (
    transcript_obs_augm["leiden_1.0_ops"].astype(float).fillna(-1).astype(str)
)

In [ ]:
(
    gg.ggplot(
        transcript_obs_augm,
        gg.aes(
            x="transcript_case_UMAP1",
            y="transcript_case_UMAP2",
        ),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.geom_point(
        transcript_obs_augm.loc[lambda x: ~x["Instantaneous Growth Rate: Volume"].isna()],
        gg.aes(color="Instantaneous Growth Rate: Volume"),
        size=1.0,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
        color="volumic growth",
    )
    + gg.theme(figure_size=(6, 4), dpi=300)
)

In [ ]:
(
    gg.ggplot(
        transcript_obs_augm,
        gg.aes(
            x="transcript_case_UMAP1",
            y="transcript_case_UMAP2",
        ),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.geom_point(
        transcript_obs_augm.loc[lambda x: ~x["Length"].isna()],
        gg.aes(color="Length"),
        size=1.0,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
    )
    + gg.theme(figure_size=(6, 4), dpi=300)
)

In [ ]:
(
    gg.ggplot(
        transcript_obs_augm,
        gg.aes(
            x="transcript_case_UMAP1",
            y="transcript_case_UMAP2",
        ),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.geom_point(
        transcript_obs_augm.loc[lambda x: ~x["Width"].isna()],
        gg.aes(color="Width"),
        size=1.0,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
    )
    + gg.theme(figure_size=(6, 4), dpi=300)
)

In [ ]:
(
    gg.ggplot(
        transcript_obs_augm,
        gg.aes(
            x="transcript_case_UMAP1",
            y="transcript_case_UMAP2",
        ),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.geom_point(
        transcript_obs_augm.loc[lambda x: ~x["Instantaneous Growth Rate: Volume"].isna()],
        gg.aes(color="Delta time (s)"),
        size=1.0,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
    )
    + gg.theme(figure_size=(6, 4), dpi=300)
)

In [ ]:
(
    gg.ggplot(
        transcript_obs_augm,
        gg.aes(
            x="transcript_case_UMAP1",
            y="transcript_case_UMAP2",
        ),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.geom_point(
        transcript_obs_augm.loc[lambda x: ~x["Instantaneous Growth Rate: Volume"].isna()],
        gg.aes(color="mCherry mean_intensity"),
        size=1.0,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
    )
    + gg.theme(figure_size=(6, 4), dpi=300)
)

In [ ]:
transcript_obs = adata_case.obs.copy()
transcript_cluster_assignment = (
    # transcript_obs.groupby("spacer")["annotated_leiden_case"]
    transcript_obs.groupby("spacer")["annotated_leiden_case_coarse"]
    .apply(lambda x: x.mode()[0])
    .to_frame("transcript_cluster_assignment")
    .reset_index()
)
transcript_cluster_assignment

In [ ]:
ops_df_ = ops_df.merge(
    transcript_cluster_assignment, left_on="sgRNA", right_on="spacer", how="left"
)
opssummary_df_ = opssummary_df.merge(
    transcript_cluster_assignment, left_index=True, right_on="spacer", how="left"
)

In [ ]:
opssummary_df_

In [ ]:
fig = (
    gg.ggplot(opssummary_df_, gg.aes("opssummary_TSNE1", "opssummary_TSNE2"))
    + gg.geom_point(size=0.2)
    + gg.geom_point(
        opssummary_df_.loc[lambda x: ~x["transcript_cluster_assignment"].isna()],
        gg.aes(color="transcript_cluster_assignment"),
        size=2.0,
    )
    + gg.theme_minimal()
    + gg.theme(
        # legend_position="none",
        figure_size=(8, 5),
    )
    + gg.labs(
        x="t-SNE 1",
        y="t-SNE 2",
        title="OPS representation",
    )
    + gg.scale_color_manual(tab10_hex)
    # + gg.scale_color_cmap("bwr", limits=[-3.0, 3.0])
)
fig

In [ ]:
fig = (
    gg.ggplot(ops_df_, gg.aes("tsne_x", "tsne_y"))
    + gg.geom_point(size=0.2)
    + gg.geom_point(
        ops_df_.loc[lambda x: ~x["transcript_cluster_assignment"].isna()],
        gg.aes(color="transcript_cluster_assignment"),
        size=2.0,
    )
    + gg.theme_minimal()
    + gg.theme(
        # legend_position="none",
        figure_size=(8, 5),
    )
    + gg.labs(
        x="t-SNE 1",
        y="t-SNE 2",
        title="OPS representation",
    )
    + gg.scale_color_manual(tab10_hex)
    # + gg.scale_color_cmap("bwr", limits=[-3.0, 3.0])
)
fig

### Joint embeddings

#### Approach 0: capsule-level (no data removal)

In [ ]:
adata_case_joint = adata_case.copy()
adata_case_joint.obs = adata_case_joint.obs.merge(
    ops_df, left_on="spacer", right_on="sgRNA", how="left", suffixes=("_tr", "_ops")
)

below is an unsuccessful approach: replacing missing values with 0.0 does not suffice to make appear useful variations

In [ ]:
# X_ops = []

# default_ops_vals = np.zeros(len(f_vector_cols))
# ops_df_ = ops_df.copy().loc[:, f_vector_cols]
# ops_df_ = ops_df_.fillna(0.0)
# ops_df_ = ops_df_ - ops_df_.mean(axis=0)
# ops_df_ = ops_df_ / ops_df_.std(axis=0)

# for obs in adata_case.obs.index:
#     guide = adata_case.obs.loc[obs, "spacer"]
#     if guide not in ops_df["sgRNA"].unique():
#         X_ops.append(default_ops_vals)
#     else:
#         X_ops.append(ops_df_.loc[ops_df["sgRNA"] == guide].values.flatten())
# X_ops = np.stack(X_ops)

# adata_case.obsm["X_ops"] = X_ops
# adata_case.obsm["X_joint"] = np.concatenate([adata_case.obsm["X_scVI"], X_ops], axis=1)
# sc.pp.neighbors(adata_case, use_rep="X_joint")
# sc.tl.umap(adata_case)
# sc.pl.umap(adata_case, color=["annotated_leiden_case"])

In [ ]:
ops_df_ = ops_df.copy().set_index("sgRNA").loc[:, f_vector_cols]
ops_df_ = ops_df_.fillna(0.0)
ops_df_ = ops_df_ - ops_df_.mean(axis=0)
ops_df_ = ops_df_ / ops_df_.std(axis=0)
ops_df_

In [ ]:
ops_df

In [ ]:
ops_df.set_index("sgRNA").loc[adata_case_train.obs["spacer"]]

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

overlapping_spacers = np.intersect1d(
    ops_df["sgRNA"].unique().astype(str), adata_case_joint.obs["spacer"].unique().astype(str)
)
adata_case_train = adata_case[adata_case.obs["spacer"].isin(overlapping_spacers)].copy()
y_train = (
    ops_df.set_index("sgRNA")
    .loc[adata_case_train.obs["spacer"], f_vector_cols]
    .fillna(ops_df[f_vector_cols].mean())
)
X_train = adata_case_train.obsm["X_scVI"]

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(adata_case_joint.obsm["X_scVI"])
adata_case_joint.obsm["X_ops_pred"] = y_pred
adata_case_joint.obsm["X_joint_pred"] = np.concatenate(
    [adata_case_joint.obsm["X_scVI"], y_pred], axis=1
)
adata_case_joint.obsm["X_joint_pred_normalized"] = (
    adata_case_joint.obsm["X_joint_pred"] - adata_case_joint.obsm["X_joint_pred"].mean(axis=0)
) / adata_case_joint.obsm["X_joint_pred"].std(axis=0)
new_f_names = ["{}_pred".format(c) for c in f_vector_cols]
adata_case_joint.obs[new_f_names] = adata_case_joint.obsm["X_ops_pred"]

In [ ]:
sc.pp.neighbors(adata_case_joint, use_rep="X_scVI")
sc.tl.umap(adata_case_joint)
sc.pl.umap(adata_case_joint, color=["annotated_leiden_case"])
plt.show()
for c in f_vector_cols:
    sc.pl.umap(adata_case_joint, color=[c, "{}_pred".format(c)])
    plt.show()

In [ ]:
plot_df = adata_case.obs.copy()
plot_df["UMAP1"] = adata_case.obsm["X_umap"][:, 0]
plot_df["UMAP2"] = adata_case.obsm["X_umap"][:, 1]
fig = px.scatter(
    plot_df,
    x="UMAP1",
    y="UMAP2",
    color="annotated_leiden_case",
    hover_data=["target"],
    title="Interactive UMAP: Annotated Leiden Case",
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
sc.pp.neighbors(adata_case_joint, use_rep="X_joint_pred_normalized")
sc.tl.umap(adata_case_joint, key_added="umap_joint_pred_normalized")
# sc.pl.umap(adata_case_joint, color=["annotated_leiden_case"] + f_vector_cols, vmax="p95", vmin="p5")
sc.pl.embedding(
    adata_case_joint, color=["annotated_leiden_case"], basis="umap_joint_pred_normalized"
)
adata_case_joint.obs["joint_case_UMAP1"] = adata_case_joint.obsm["umap_joint_pred_normalized"][:, 0]
adata_case_joint.obs["joint_case_UMAP2"] = adata_case_joint.obsm["umap_joint_pred_normalized"][:, 1]

In [ ]:
plot_df = adata_case_joint.obs.copy()
fig = px.scatter(
    plot_df,
    x="joint_case_UMAP1",
    y="joint_case_UMAP2",
    color="annotated_leiden_case",
    hover_data=["target"] + f_vector_cols,
    title="Interactive UMAP: Annotated Leiden Case",
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
sc.tl.leiden(adata_case_joint, key_added="leiden_joint_pred", resolution=0.1)
sc.pl.umap(adata_case_joint, color=["leiden_joint_pred"])

In [ ]:
cluster_count = adata_case_joint.obs.query("leiden_joint_pred == '4'")[
    "annotated_leiden_case"
].value_counts()
cluster_count = cluster_count[cluster_count >= 10]
clusters_to_analyze = cluster_count.index.values
clusters_to_analyze

In [ ]:
for cluster in clusters_to_analyze:
    print(cluster)
    obs_subset = adata_case_joint.obs.loc[
        # lambda x: x["annotated_leiden_case"].isin(clusters_to_analyze)
        lambda x: x["annotated_leiden_case"]
        == cluster
    ]
    gp1 = obs_subset.loc[lambda x: x["leiden_joint_pred"] == "4"]
    gp2 = obs_subset.loc[lambda x: x["leiden_joint_pred"] != "4"]

    genes1 = gp1["target"].value_counts().loc[lambda x: x >= 2].index.values
    print("group 1: ", ", ".join(genes1))
    genes2 = gp2["target"].value_counts().loc[lambda x: x >= 2].index.values
    print("group 2: ", ", ".join(genes2))
    print()
    break

In [ ]:
names1 = ["aroK", "cbrC"]
names2 = ["cyoA", "hemE", "ubiG", "tufA", "cyoB", "cyoD", "brnQ", "cyoC"]

obs_genes_of_interest = adata_case_joint.obs.loc[lambda x: x["target"].isin(names1 + names2)]
obs_genes_of_interest["gene_type"] = np.where(
    obs_genes_of_interest["target"].isin(gp1),
    "respiratory",
    "biosynthetic",
)

fig_trans = (
    gg.ggplot(
        adata_case_joint.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.theme_minimal()
    + gg.geom_point(
        obs_genes_of_interest,
        gg.aes(color="gene_type"),
        size=2.0,
    )
)
fig_joint = (
    gg.ggplot(
        adata_case_joint.obs,
        gg.aes(x="joint_case_UMAP1", y="joint_case_UMAP2"),
    )
    + gg.geom_point(color="gray", size=0.5)
    + gg.theme_minimal()
    + gg.geom_point(obs_genes_of_interest, gg.aes(color="gene_type"), size=2.0)
)

display(fig_trans)
display(fig_joint)

In [ ]:
gp1

#### Approach 1: spacer-level (unsucessful)

In [ ]:
ops_df["sgRNA"].unique()

In [ ]:
# extract avg scVI rep per guide
_trans_embeddings = []
trans_spacers = adata_case.obs["spacer"].unique().categories.values
for spacer in trans_spacers:
    adata_case_sub = adata_case[adata_case.obs["spacer"] == spacer].copy()
    _trans_embeddings.append(adata_case_sub.obsm["X_scVI"].mean(axis=0))
    # _trans_embeddings.append(adata_case_sub.obsm["X_scVI"][0])
_trans_embeddings = np.stack(_trans_embeddings)

valid_spacers = np.intersect1d(trans_spacers, ops_df["sgRNA"].unique())
print("total # of spacers:", len(trans_spacers))
print("total # of valid spacers:", len(valid_spacers))
trans_embeddings_df = pd.DataFrame(_trans_embeddings, index=trans_spacers)
trans_embeddings = trans_embeddings_df.loc[valid_spacers].values

# f_vector_cols = [c for c in ops_df.columns if c.startswith("Feature Vector")]
f_vector_cols = [
    "N Mismatch_y",
    "Delta time (s)",
    "Instantaneous Growth Rate: Volume",
    "Length",
    "Septum Displacement Length Normalized",
    "Width",
    "mCherry mean_intensity",
]
_ops_embeddings_df = ops_df.set_index("sgRNA")[f_vector_cols].loc[valid_spacers]
_ops_embeddings = _ops_embeddings_df.fillna(_ops_embeddings_df.mean()).values
# pca_ = PCA(n_components=10)
# ops_embeddings = pca_.fit_transform(_ops_embeddings)
ops_embeddings = _ops_embeddings

In [ ]:
joint_embeddings = trans_embeddings
# joint_embeddings = np.concatenate([trans_embeddings, ops_embeddings], axis=1)
# joint_embeddings = (joint_embeddings - joint_embeddings.mean(axis=0)) / joint_embeddings.std(axis=0)

tsne_ = TSNE(n_components=2, perplexity=100, max_iter=1000)
embeddings_tsne = tsne_.fit_transform(joint_embeddings)

adata_mock = sc.AnnData(X=joint_embeddings, obsm=dict(joint_embeddings=joint_embeddings))
sc.pp.neighbors(adata_mock, use_rep="joint_embeddings")
sc.tl.umap(adata_mock)

embeddings_umap = adata_mock.obsm["X_umap"]


plot_df = (
    pd.DataFrame(embeddings_tsne, index=valid_spacers, columns=["joint_tsne_x", "joint_tsne_y"])
    .assign(joint_umap_x=embeddings_umap[:, 0], joint_umap_y=embeddings_umap[:, 1])
    .merge(transcript_obs_augm, left_index=True, right_on="spacer", how="left")
)

(
    gg.ggplot(plot_df, gg.aes("joint_umap_x", "joint_umap_y", color="annotated_leiden_case"))
    + gg.geom_point()
    + gg.theme_minimal()
    + gg.theme(figure_size=(10, 4), dpi=100)
)